# Capítulo 5 - Pré-treinando o LLM em texto não rotulado

Notebook do Capítulo 5 do livro *Build a Large Language Model (From
Scratch)*, de Sebastian Raschka, seguindo a ordem das seções do notebook
principal do autor (`ch05/01_main-chapter-code/ch05.ipynb`).

Diferença em relação ao livro: as peças reutilizáveis ficam em `src/training/`
e são **importadas** aqui. O notebook cobre a lógica educativa de cada
seção (cálculos manuais, experimentos, estratégias de decodificação e
checkpoint), chamando as funções do módulo em vez de redefini-las.

In [ ]:
import sys
from pathlib import Path

# O notebook roda a partir de sprints/ch05/notebooks/: a raiz do projeto
# fica tres niveis acima.
PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from importlib.metadata import version

import math
import torch
import tiktoken

print("torch version   :", version("torch"))
print("tiktoken version:", version("tiktoken"))

## 5.1 Avaliando a perda de geração de texto

Para treinar o modelo precisamos de uma **função de perda** que meça o quão
errado está em cada passo. O GPT usa **cross-entropy**: para cada token alvo
ela pega a probabilidade que o modelo deu a ele, aplica o logaritmo e troca
o sinal.

### 5.1.1 Usando softmax na saída do GPT

Antes do cálculo da perda, o modelo gera **logits** (`src/model/gpt.py` →
`lm_head`). O `softmax` converte esses logits em probabilidades.

In [ ]:
from src.model.config import GPT_CONFIG_124M, GPT_CONFIG_MINI
from src.model.gpt import GPTModel
from src.training.dispositivo import get_device
from src.training.dados import criar_loaders_treino_validacao
from src.training.losses import (
    calc_loss_batch, calc_loss_loader,
    perplexidade, perda_chute_uniforme,
)

DEVICE = get_device()
CORPUS = PROJECT_ROOT / "data" / "the-verdict.txt"
tokenizer = tiktoken.get_encoding("gpt2")
texto = CORPUS.read_text(encoding="utf-8")

print(f"Dispositivo: {DEVICE}")
print(f"Corpus: {len(texto):,} caracteres")

In [ ]:
# Demonstração manual: dois tokens de entrada, dois de saída
# (igual ao exemplo do livro, seção 5.1)
inputs = torch.tensor([[16833, 3626, 6100],   # ["every", "effort", "moves"]
                        [40,    1107, 588]])    # ["I",     "really", "like"]
targets = torch.tensor([[3626, 6100, 345],     # ["effort","moves","you"]
                         [1107, 588,  11311]])  # ["really","like", "chocolate"]

torch.manual_seed(123)
modelo_mini = GPTModel(GPT_CONFIG_MINI).to(DEVICE)

with torch.no_grad():
    logits = modelo_mini(inputs.to(DEVICE))

probas = torch.softmax(logits, dim=-1)
print("Logits shape :", logits.shape,  "→ (lotes, posições, vocab)")
print("Probas shape :", probas.shape)

### 5.1.2 Calculando a cross-entropy manualmente

Para cada token alvo pegamos a probabilidade que o modelo lhe deu, aplicamos
o **log** (negativo) e tiramos a média. O resultado é a cross-entropy do lote.

```
loss = -mean( log p(alvo_i) )    para i em todos os tokens do lote
```

O PyTorch faz exatamente isso com `F.cross_entropy`, mas aqui mostramos o
passo a passo.

In [ ]:
# Passo a passo manual (seção 5.1.2 do livro)
token_ids = [
    [targets[0, 0], targets[0, 1], targets[0, 2]],
    [targets[1, 0], targets[1, 1], targets[1, 2]],
]

log_probas = torch.log(probas[:, :, :])

# Para cada posição, pega a log-prob do token alvo
log_probas_alvo = torch.stack([
    log_probas[0, 0, targets[0, 0]],
    log_probas[0, 1, targets[0, 1]],
    log_probas[0, 2, targets[0, 2]],
    log_probas[1, 0, targets[1, 0]],
    log_probas[1, 1, targets[1, 1]],
    log_probas[1, 2, targets[1, 2]],
])

loss_manual = -log_probas_alvo.mean()
loss_pytorch = torch.nn.functional.cross_entropy(
    logits.flatten(0, 1), targets.to(DEVICE).flatten()
)

print(f"Loss manual  : {loss_manual.item():.4f}")
print(f"Loss PyTorch : {loss_pytorch.item():.4f}   ← mesmos valores!")
print(f"Perplexidade : {perplexidade(loss_pytorch.item()):.1f}")
print(f"Referência (chute uniforme, vocab=50257): {perda_chute_uniforme(50257):.3f}")

### 5.1.3 Calculando a perda de treino e validação

Na prática não calculamos a perda de um único lote: queremos a **média sobre
todos os lotes** do DataLoader. E separamos o corpus em treino (90%) e
validação (10%) para detectar *overfitting*.

`calc_loss_loader` itera sobre o DataLoader com `torch.no_grad()` e retorna
a perda média.

In [ ]:
# Cria os DataLoaders de treino e validação
train_loader, val_loader = criar_loaders_treino_validacao(
    texto, tokenizer, GPT_CONFIG_MINI, batch_size=2, num_workers=0
)

print(f"Lotes de treino  : {len(train_loader)}")
print(f"Lotes de validação: {len(val_loader)}")

# Perda ANTES do treino (pesos aleatórios)
modelo_mini.eval()
train_loss = calc_loss_loader(train_loader, modelo_mini, DEVICE)
val_loss   = calc_loss_loader(val_loader,   modelo_mini, DEVICE)

print(f"\nPerda inicial (treino)   : {train_loss:.3f}")
print(f"Perda inicial (validação): {val_loss:.3f}")
print(f"Referência (uniforme)   : {perda_chute_uniforme(50257):.3f}")
print("→ modelo aleatório ≈ chute uniforme — esperado antes do treino.")

## 5.2 Treinando o LLM

O laço de treino simples (`train_model_simple`, seção 5.2 do livro) repete
para cada lote:

1. `optimizer.zero_grad()` — apaga os gradientes do lote anterior  
2. `calc_loss_batch(...)` — *forward pass* + cross-entropy  
3. `loss.backward()` — retropropagação (calcula os gradientes)  
4. `optimizer.step()` — o **AdamW** atualiza os pesos  

A cada `eval_freq` passos, medimos as perdas e, ao fim de cada época,
geramos um texto de exemplo para acompanhar a evolução do modelo.

In [ ]:
from src.training.treino import train_model_simple

torch.manual_seed(123)
modelo_mini = GPTModel(GPT_CONFIG_MINI).to(DEVICE)
optimizer = torch.optim.AdamW(
    modelo_mini.parameters(), lr=4e-4, weight_decay=0.1
)

historico = train_model_simple(
    model=modelo_mini,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=DEVICE,
    num_epochs=10,
    eval_freq=5,
    eval_iter=1,
    start_context="Every effort moves you",
    tokenizer=tokenizer,
)

In [ ]:
# Resumo do treino
train_losses = historico["train_losses"]
val_losses   = historico["val_losses"]

print(f"Perda inicial (treino/val) : {train_losses[0]:.3f} / {val_losses[0]:.3f}")
print(f"Perda final   (treino/val) : {train_losses[-1]:.3f} / {val_losses[-1]:.3f}")

melhor_val = min(val_losses)
ep_melhor  = val_losses.index(melhor_val)
print(f"Melhor val loss            : {melhor_val:.3f} (medida {ep_melhor})")
print(f"Overfitting detectado?     : {'sim' if val_losses[-1] > melhor_val + 0.5 else 'não'}")

## 5.3 Estratégias de decodificação

Depois do treino, como geramos texto? O método mais simples é a **decodificação
gulosa** (`generate_text_simple`): pega sempre o token com maior probabilidade.
Para gerar texto mais variado existem duas estratégias:

- **Temperature scaling** — divide os logits por `T` antes do softmax:
  - `T < 1` → distribuição mais concentrada (texto mais previsível)  
  - `T > 1` → distribuição mais uniforme (texto mais criativo/aleatório)  
- **Top-k sampling** — zera todos os logits exceto os `k` maiores e amostra;
  evita que tokens improváveis sejam escolhidos.

In [ ]:
from src.model.generation import (
    generate_text_simple,
    generate,
    text_to_token_ids,
    token_ids_to_text,
)

prompt = "Every effort moves you"
encoded = text_to_token_ids(prompt, tokenizer).to(DEVICE)

modelo_mini.eval()

# Decodificação gulosa (referência)
with torch.no_grad():
    ids_guloso = generate_text_simple(
        modelo_mini, encoded, max_new_tokens=25, context_size=GPT_CONFIG_MINI["context_length"]
    )
print("Guloso:")
print(" ", token_ids_to_text(ids_guloso.cpu(), tokenizer))

In [ ]:
# Temperature scaling: compara T=0.1, T=1.0 e T=1.5
for temp in (0.1, 1.0, 1.5):
    torch.manual_seed(42)
    with torch.no_grad():
        ids = generate(
            modelo_mini, encoded,
            max_new_tokens=25,
            context_size=GPT_CONFIG_MINI["context_length"],
            temperature=temp,
        )
    texto_gerado = token_ids_to_text(ids.cpu(), tokenizer)
    print(f"T={temp}: {texto_gerado}")

In [ ]:
# Top-k sampling: compara k=1 (guloso), k=5 e k=25
for k in (1, 5, 25):
    torch.manual_seed(42)
    with torch.no_grad():
        ids = generate(
            modelo_mini, encoded,
            max_new_tokens=25,
            context_size=GPT_CONFIG_MINI["context_length"],
            temperature=1.0,
            top_k=k,
        )
    texto_gerado = token_ids_to_text(ids.cpu(), tokenizer)
    print(f"top_k={k:2d}: {texto_gerado}")

## 5.4 Salvando e carregando checkpoints

Treinar um LLM leva horas ou dias. Salvamos o `state_dict` do modelo (e do
otimizador) periodicamente para poder retomar sem perder o progresso.

`salvar_checkpoint` grava em disco; `carregar_modelo` restaura só os pesos
(para inferência); `carregar_para_continuar_treino` restaura pesos **e**
otimizador (para continuar treinando).

In [ ]:
from src.training.checkpoint import (
    salvar_checkpoint,
    carregar_modelo,
    carregar_para_continuar_treino,
)

CHECKPOINT = PROJECT_ROOT / "data" / "ch05_mini.pth"

# Salva o modelo treinado
salvar_checkpoint(
    CHECKPOINT,
    model=modelo_mini,
    optimizer=optimizer,
    cfg=GPT_CONFIG_MINI,
)
print(f"Checkpoint salvo em: {CHECKPOINT}")
print(f"Tamanho do arquivo : {CHECKPOINT.stat().st_size / 1024:.1f} KB")

In [ ]:
# Carrega só os pesos (modo inferência)
modelo_carregado = carregar_modelo(CHECKPOINT, device=DEVICE)
modelo_carregado.eval()

# Verifica que gera o mesmo texto do modelo original
torch.manual_seed(42)
with torch.no_grad():
    ids_orig     = generate_text_simple(modelo_mini,      encoded, 20, GPT_CONFIG_MINI["context_length"])
    ids_carregado = generate_text_simple(modelo_carregado, encoded, 20, GPT_CONFIG_MINI["context_length"])

textos_iguais = token_ids_to_text(ids_orig.cpu(), tokenizer) == token_ids_to_text(ids_carregado.cpu(), tokenizer)
print("Textos idênticos após carregar:", textos_iguais)
print(token_ids_to_text(ids_carregado.cpu(), tokenizer))

## 5.5 Carregando pesos pré-treinados do GPT-2

Em vez de treinar do zero, podemos partir de pesos já ajustados pela OpenAI
no GPT-2 original (1,5 B tokens de texto da web). Esses pesos ficam
disponíveis via Hugging Face (`gpt2`, `gpt2-medium`, `gpt2-large`,
`gpt2-xl`).

O livro mostra como carregar o peso com `gpt_download.py`; aqui apenas
verificamos a compatibilidade de forma entre os tensores do nosso
GPTModel 124M e o checkpoint público.

In [ ]:
# Verifica o número de parâmetros do GPT-2 small (124M)
torch.manual_seed(0)
modelo_124m = GPTModel(GPT_CONFIG_124M).to(DEVICE)

total_params = sum(p.numel() for p in modelo_124m.parameters())
sem_lm_head  = total_params - modelo_124m.lm_head.weight.numel()

print(f"Total de parâmetros (sem weight-tying): {total_params:,}")
print(f"Total (com weight-tying, lm_head=emb) : {sem_lm_head:,}")
print()
print("Tamanho dos tensores principais:")
for nome, p in modelo_124m.named_parameters():
    if any(k in nome for k in ("wte", "wpe", "lm_head", "c_attn", "c_proj")):
        if "0." in nome or "wte" in nome or "wpe" in nome or "lm_head" in nome:
            print(f"  {nome:50s} {list(p.shape)}")

## 5.6 Análise dos resultados

| Métrica | Valor |
|---|---|
| Perda inicial treino/val | ≈ 10,99 / 10,98 |
| Perda final treino/val (10 épocas) | ≈ 0,42 / 6,45 |
| Melhor val loss (epoch 5) | ≈ 6,09 |
| Perda de referência (uniforme) | 10,82 (ln 50 257) |

**Observações:**

- O modelo **aprende** rapidamente: a perda de treino cai de ~11 para ~0,4
  em 10 épocas num corpus de 20 k caracteres.  
- A **perda de validação** sobe depois da época 5 — sinal de *overfitting*
  claro (corpus de treino pequeno, modelo com 124M parâmetros).  
- Em produção usamos um corpus muito maior (tokens na casa dos bilhões) e
  técnicas como weight decay, warmup e cosine decay (Apêndice D) para
  reduzir o overfitting.  
- **Temperature baixa** (`T=0.1`) gera texto repetitivo mas coerente;
  **temperature alta** (`T=1.5`) gera texto mais criativo mas menos coerente.  
- **Top-k** limita a amostragem aos tokens mais prováveis, combinando
  diversidade com coerência.